<a href="https://colab.research.google.com/github/ipr2104/Advanced-Quant/blob/main/Reference_Quiz_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up and loading data

In [ ]:
#Imports
import numpy as np                              # arrays and math
import pandas as pd                             # tables (DataFrames)
import matplotlib.pyplot as plt                 # plots
from scipy import stats, signal                 # tests, distributions, filters, spectra
import statsmodels.api as sm                    # regression (sm.OLS)
import statsmodels.tsa.stattools as stattools   # stattools.acf, stattools.acovf
from statsmodels.tsa.stattools import acf       # same acf, shorter name (HW3, Lecture 10)
import xarray as xr

In [ ]:
#Load csv
df = pd.read_csv("n34_all.csv", parse_dates=["time"])

#Add year and month
df["year"]  = df["time"].dt.year
df["month"] = df["time"].dt.month

#Convert date if needed
d["Date"] = pd.to_datetime(d["Date"], format="%Y-%m")

In [ ]:
#Clean and sort data (drop NAs and sort)
df = df.dropna(subset=["TMAX"]).sort_values("DATE").reset_index(drop=True)

#Convert units for temp
df["TMAX_C"] = (df["TMAX"] - 32) * 5/9      # Fahrenheit to Celsius

In [ ]:
#Average by group
#Monthly mean from daily data
monthly = df.groupby(["Year", "Month"])["TMAX"].mean().reset_index()

#Annual mean temperature and annual total precipitation (Lecture 3)
annual_temp = df.groupby("year")["tmean"].mean()
annual_ppt  = df.groupby("year")["ppt"].sum()

- groupby splits the table into pieces (one per year, or per year-and-month), applies .mean() or .sum() to each piece, and puts the results back together.
-  Use .mean() for temperature and .sum() for precipitation, because precipitation accumulates.
- .reset_index() turns the group labels back into ordinary columns.

In [ ]:
#Cut two time periods
early  = df.loc[(df["year"] >= 1951) & (df["year"] <= 1980), "nino34"].values
recent = df.loc[(df["year"] >= 1996) & (df["year"] <= 2025), "nino34"].values

In [ ]:
#Sort by season (DJF)
djf = df[df["month"].isin([12, 1, 2])].copy()
djf["djf_year"] = djf["year"]
djf.loc[djf["month"] == 12, "djf_year"] = djf["year"] + 1   # Dec belongs to the following winter
counts = djf.groupby("djf_year")["nino34"].count()
complete_years = counts[counts == 3].index                  # keep only winters with all 3 months
djf_seasonal = (djf[djf["djf_year"].isin(complete_years)]
                .groupby("djf_year")["nino34"].mean().reset_index())

In [ ]:
#Training and testing (HW 3)
train = np.arange(27)          # rows 0 to 26 = 1984 to 2010
test  = np.arange(27, n)       # rows 27 to 36 = 2011 to 2020, where n = len(df)

In [ ]:
#Lecture 2 Nino 3.4 from gridded data
# anomaly = value minus that calendar month's 1991-2020 average
monthly_mean_sst = sst.sel(time=slice('1991', '2020')).groupby('time.month').mean('time')
ssta = sst.groupby('time.month') - monthly_mean_sst

def n34_average(x):
    x = x.sortby("latitude")
    weights = np.cos(np.deg2rad(x.latitude))                 # grid cells shrink toward the poles
    y = (x.sel(longitude=slice(190, 240)).sel(latitude=slice(-5, 5))
          .weighted(weights).mean(['longitude', 'latitude']))
    return y

n34_a = n34_average(ssta)

- The anomaly step subtracts each calendar month's 30-year average, so a July value is compared with other Julys.
- The Niño 3.4 box is 5°N to 5°S and 170°W to 120°W. Longitude here runs 0 to 360, so 170°W = 190 and 120°W = 240.
- Weighting by the cosine of latitude fixes the fact that a 2° by 2° cell near the equator covers more area than one near the poles.
- The notebook downloads the file with !curl, which works only in Colab (the ! runs a shell command).

# Describing data

In [ ]:
#Moments and quantiles (from lecture 2)
np.mean(x)                          # sample mean
np.std(x)                           # standard deviation (divides by n)
stats.skew(x)                       # skewness
stats.kurtosis(x)                   # EXCESS kurtosis: a normal gives 0
stats.kurtosis(x, fisher=False)     # your notes' definition: a normal gives 3
np.percentile(x, [5, 25, 50, 75, 95])

In [ ]:
#Histogram and QQ plot
plt.hist(x)                                   # shape of the distribution
plt.xlabel("Anomaly, degrees C"); plt.ylabel("Frequency")

stats.probplot(x, dist="norm", plot=plt)      # QQ plot against a normal (added)

In [ ]:
#Fit gamma distribution to rainfall
df['PRCP'] = pd.to_numeric(df['PRCP'], errors='coerce')
df = df.dropna(subset=['PRCP'])
wet = df.loc[df['PRCP'] > 0, 'PRCP'].values            # keep wet days only
frac_wet = len(wet) / len(df)

shape, loc, scale = stats.gamma.fit(wet, floc=0)       # k = shape, theta = scale
print(shape * scale, shape * scale**2)                 # implied mean and variance

In [ ]:
#Probability of exceedance and return period (Is July 2026 unusually wet?)
monthly = df.set_index('DATE')['PRCP'].resample('MS').sum()    # monthly totals
monthly = monthly[monthly.index < monthly.index.max()]          # drop the incomplete last month

july = monthly[monthly.index.month == 7].sort_values()
val_2026 = july[july.index.year == 2026].values[0]

shape, loc, scale = stats.gamma.fit(july.values, floc=0)
fit_p = 1 - stats.gamma.cdf(val_2026, shape, loc, scale)       # P(a July is wetter than this one)
fit_return_period = 1 / fit_p

- resample('MS').sum() adds up daily rainfall into month totals ('MS' means month start).
- The CDF gives the probability of being at or below a value, so 1 - cdf is the probability of exceedance.
- The return period is 1 divided by that probability. Your notebook got 2.36, which your notes round to 2.4: a July this wet happens about every 2 to 3 years, so it is not remarkable.

# Hypothesis tests

In [ ]:
#Correlation between two variables
rho, p = stats.pearsonr(x, y)        # linear relationship
rs = stats.spearmanr(x, y)           # monotonic relationship, uses ranks (returns statistic and pvalue)

In [ ]:
#T test for mean
t_stat, p_val = stats.ttest_ind(recent, early, equal_var=False)
print(f"Welch's t-test: {t_stat:.2f}; p-value: {p_val:.3g}")

In [ ]:
#F test for variance
def f_test(x1, x2):
    f_stat = x2.var(ddof=1) / x1.var(ddof=1)               # ratio of the two sample variances
    dfn, dfd = len(x2) - 1, len(x1) - 1                    # degrees of freedom
    p_f = 2 * min(stats.f.cdf(f_stat, dfn, dfd),
                  1 - stats.f.cdf(f_stat, dfn, dfd))       # two-sided p-value
    return f_stat, dfn, dfd, p_f

In [ ]:
#Correlation tests
rho, p_scipy = stats.pearsonr(x, y)
stats.spearmanr(annual_temp.values, annual_ppt.values)

In [ ]:
#Non parametric tests
z_stat, p_val = stats.ranksums(early, recent)                          # Wilcoxon rank-sum
u_stat, p_value = stats.mannwhitneyu(early, recent, alternative='two-sided')   # same idea, continuity-corrected
D_stat, p_val = stats.ks_2samp(early, recent)                          # Kolmogorov-Smirnov
H_stat, p_value = stats.kruskal(jfm_series, amj_series, jas_series, ond_series)   # 3+ groups

-  Rank-sum replaces values by their ranks (1 for the smallest) and asks whether one group's ranks run systematically higher. It does not test means. mannwhitneyu is the same test with a continuity correction, so its p-value differs slightly (0.947 versus 0.941).
- KS compares the whole distribution: it builds each sample's empirical CDF and reports D, the largest vertical gap between them. D near 0 means similar distributions.
- Kruskal-Wallis extends rank-sum to three or more groups. A large H and a small p-value say that at least one group differs; it does not say which. Your seasonal Oregon precipitation test gave H = 411.8 and p = 6e-89.
- A p-value around 0.9, as in the Niño 3.4 winter tests, means no evidence of a difference: fail to reject H0.

In [ ]:
#Selecting a season series for these tests (use the strings in the file):
amj_series = seasonal.loc[seasonal["season"] == "AMJ", "tmean (degrees F)"]
ond_series = seasonal.loc[seasonal["season"] == "OND", "tmean (degrees F)"]
D_stat, p_val = stats.ks_2samp(amj_series.values, ond_series.values)

In [ ]:
#Permutation test
Nx, Ny = len(early), len(recent)
combined = np.concatenate([early, recent])
z_obs, p_asymp = stats.ranksums(early, recent)       # the statistic you actually observed

perm_z = np.zeros(500)
for i in range(500):
    shuffled = np.random.permutation(combined)       # shuffle all labels
    g1, g2 = shuffled[:Nx], shuffled[Nx:]            # re-split into two fake groups
    perm_z[i], _ = stats.ranksums(g1, g2)            # recompute the statistic

p_perm = np.mean(np.abs(perm_z) >= np.abs(z_obs))    # share of shuffles at least as extreme

In [ ]:
#Hurricane counts by ENSO phase (Lecture 4 example)
el_nino = tc.loc[tc["enso_phase"] == "El Nino", "hurricane_count"]
la_nina = tc.loc[tc["enso_phase"] == "La Nina", "hurricane_count"]
t_stat, p_val = stats.ttest_ind(el_nino, la_nina, equal_var=False)

# Autocorrelation and effective N

From Lectures 5 and 7. Correlated data carry less information than the same number of independent points, so you shrink N before computing a standard error or p-value. This section is the most reused idea in the course: it appears again in the trend, regression and attribution work

In [ ]:
#Autocovariance and ACF
roni = df["RONI"].values
n = len(roni)
max_lag = 10

c   = stattools.acovf(roni, adjusted=False, nlag=max_lag)     # autocovariance at lags 0 to 10
rho = stattools.acf(roni, nlags=max_lag, adjusted=False)      # ACF = autocovariance / variance

#Your notebook also writes the same thing by hand:
def sample_autocovariance(x, k):
    n = len(x)
    xc = x - np.mean(x)                         # subtract the mean
    return np.sum(xc[:n - k] * xc[k:]) / n      # multiply each value by the one k steps later, average

def sample_acf(x, max_lag):
    gamma0 = sample_autocovariance(x, 0)        # this is the variance
    return np.array([sample_autocovariance(x, k) / gamma0 for k in range(max_lag + 1)])

In [ ]:
#Significance bounds and effective sample size
ci_naive = 1.96 / np.sqrt(n)                         # bound if all n points were independent
n_eff    = n * (1 - rho[1]) / (1 + rho[1])           # effective number of independent points
ci_neff  = 1.96 / np.sqrt(n_eff)                     # wider bound that accounts for persistence

plt.bar(np.arange(0, max_lag + 1), rho, width=0.6)
plt.axhline(ci_naive, color="gray", linestyle="--")
plt.axhline(ci_neff,  color="r",    linestyle="--")

- An ACF bar that sticks out beyond the bound is significantly different from zero. Persistence lasts until the bars drop inside the bound.
- N_eff uses the lag-1 value: the more persistent the series, the fewer independent points it holds. With ρ(1) = 0 you get N_eff = N; with ρ(1) = 0.7 you get about 17% of N.
- Your results: RONI (n = 916) has N_eff = 15.3 and the bound grows from ±0.065 to ±0.50. NYC daily Tmax anomaly (n = 2192) has N_eff = 507.5 and the bound grows from ±0.042 to ±0.087.

In [ ]:
#Two-sample t-test corrected for autocorrelation (Lecture 5)
rho1_early  = stattools.acf(early,  nlags=1, adjusted=False)[1]
rho1_recent = stattools.acf(recent, nlags=1, adjusted=False)[1]
n_eff1 = n1 * (1 - rho1_early)  / (1 + rho1_early)      # n1, n2 = len(early), len(recent)
n_eff2 = n2 * (1 - rho1_recent) / (1 + rho1_recent)

se1 = s1 / np.sqrt(n_eff1)                              # standard error of each mean, with N_eff in place of n
se2 = s2 / np.sqrt(n_eff2)
se_diff = np.sqrt(se1**2 + se2**2)
t_corrected = (mean2 - mean1) / se_diff

# Welch-Satterthwaite degrees of freedom, using N_eff
df_corrected = (se1**2 + se2**2)**2 / ((se1**2)**2/(n_eff1 - 1) + (se2**2)**2/(n_eff2 - 1))
p_corrected = 2 * (1 - stats.t.cdf(abs(t_corrected), df_corrected))

In [ ]:
#One-sample t-test corrected for autocorrelation (NYC Tmax anomaly)
t_stat, p_value = stats.ttest_1samp(anomaly, popmean=0)     # naive version

xbar = anomaly.mean()
s = anomaly.std(ddof=1)
se_corrected = s / np.sqrt(n_eff)
t_corrected  = xbar / se_corrected
df_corrected = n_eff - 1
p_corrected  = 2 * (1 - stats.t.cdf(abs(t_corrected), df_corrected))

#This tests whether the mean anomaly differs from 0

In [ ]:
#Confidence interval for a trend (Lecture 7 and HW2)
anomaly = df["Anomaly"].values                  # Y
n = len(anomaly)
t = np.arange(n)                                # X: time in months

slope, intercept = np.polyfit(t, anomaly, 1)    # straight-line fit
trend = intercept + slope * t
resid = anomaly - trend                         # detrended series

sigma_hat = np.sqrt(np.sum(resid**2) / (n - 2))              # residual standard deviation
SE_b = sigma_hat / np.sqrt(np.sum((t - t.mean())**2))        # standard error of the slope

rho1  = stattools.acf(resid, nlags=1, adjusted=False)[1]     # persistence of the RESIDUALS
n_eff = n * (1 - rho1) / (1 + rho1)
SE_eff = SE_b * np.sqrt(n / n_eff)                           # inflate the standard error
t_crit = stats.t.ppf(1 - 0.05/2, n_eff - 2)                  # t critical value with N_eff

ci = t_crit * SE_eff
print(f"warming rate: {slope*12:.4f} +/- {ci*12:.4f} C/year")   # x12: slope is per month

- np.polyfit(t, y, 1) fits y = intercept + slope × t and returns the slope first. The residuals are what is left after removing the trend.
- The slope's standard error is the residual spread divided by how spread out the time axis is.
- Units: the slope is per time step (month here), so multiply by 12 for per year, or by 120 for per decade.
- Autocorrelation is measured on the residuals, not on the raw series. Inflate the standard error by the square root of n / N_eff and use a t critical value with N_eff − 2 degrees of freedom.
- Your NASA GISTEMP result: slope 0.0235 °C/year, 95% CI ±0.0014 treating points as independent but ±0.0032 with N_eff, so the rate is 0.0204 to 0.0267 °C/year. In the plot, fill_between draws the band around the fitted line.

In [ ]:
#HW2 does the same job with linregress:
res = stats.linregress(t_years, y)               # slope, intercept, stderr, ...
resid = y - (res.intercept + res.slope * t_years)
r1 = acf(resid, nlags=1, adjusted=False)[1]
n_eff = n * (1 - r1) / (1 + r1)
se_adj = res.stderr * np.sqrt((n - 2) / (n_eff - 2))
tcrit_adj = stats.t.ppf(0.975, n_eff - 2)
ci_adj = (res.slope - tcrit_adj * se_adj, res.slope + tcrit_adj * se_adj)

# AR(1) models and forecast intervals

An AR(1) model predicts each value as a fraction φ of the previous one plus random noise. Because the noise is random, the forecast is a probability distribution, not a single number.

In [ ]:
#Fit the model
#RONI is labeled by 3-month season (DJF, JFM, …), so the notebook first converts labels to dates:
seasons = ["DJF","JFM","FMA","MAM","AMJ","MJJ","JJA","JAS","ASO","SON","OND","NDJ"]
season_to_month = {s: i+1 for i, s in enumerate(seasons)}          # DJF -> 1, JFM -> 2, ...
df["month_idx"] = df["Season"].map(season_to_month)
df["date"] = pd.to_datetime(df["Year"].astype(str) + "-" + df["month_idx"].astype(str) + "-01")

roni = df["RONI"].values
rho = stattools.acf(roni, nlags=1, adjusted=False)
phi_hat = rho[1]                         # for an AR(1) the lag-1 autocorrelation IS phi
x_hat = phi_hat * roni[:-1]              # prediction of each month from the previous month
residuals = roni[1:] - x_hat             # observed minus predicted = the noise W_t
sigma_W_hat = residuals.std()            # size of the noise (0.2095 in your notebook)

In [ ]:
#Forecast distribution from a starting value
idx = np.where(df['date'] == '1997-07-01')[0][0]      # position of the starting month
x0 = roni[idx]
forecast_horizon = 12                                 # months ahead
t_ahead = np.arange(0, forecast_horizon + 1)          # 0, 1, ..., 12

x_hat_mean = phi_hat**t_ahead * x0
x_hat_var  = sigma_W_hat**2 * (1 - phi_hat**(2*t_ahead)) / (1 - phi_hat**2)
x_hat_var[0] = 0                                      # at t = 0 the value is known
x_hat_std = np.sqrt(x_hat_var)

dates = df['date'][idx:idx+forecast_horizon+1]
plt.plot(dates, roni[idx:idx+forecast_horizon+1], 'o-')           # what really happened
plt.plot(dates, x_hat_mean, 'o-')                                 # forecast
plt.fill_between(dates, x_hat_mean - 1.96*x_hat_std, x_hat_mean + 1.96*x_hat_std,
                 color='orange', alpha=0.2, label='95% forecast interval')

In [ ]:
#Does AR(1) reproduce the real series?
max_lag = 12
rho = stattools.acf(roni, nlags=max_lag, adjusted=False)[1:]       # observed ACF, lags 1 to 12
acf_model = phi_hat ** np.arange(1, max_lag + 1)                   # AR(1) theory: rho at lag k is phi^k

plt.bar(np.arange(1, max_lag+1), rho, label='Observed sample ACF')
plt.plot(np.arange(1, max_lag+1), acf_model, 'ko-', label='Fitted AR(1) theoretical ACF')
plt.legend()

The bars are the data and the line is what an AR(1) with your fitted φ predicts. A big mismatch means a high lag-1 autocorrelation did not make AR(1) a good model for the whole series, which is the Lecture 6 conclusion for ENSO (an AR(1) cannot capture periodic behavior).

In [ ]:
#Hands-on: NYC daily Tmax anomaly
x = df.anomaly.values
rho = stattools.acf(x, nlags=1, adjusted=False)[1]
x_hat = rho * x[:-1]
residuals = x[1:] - x_hat
sigma_w_hat = residuals.std()

idx0 = np.where(df["DATE"] == "2000-01-01")[0][0]
x0 = x[idx0]
forecast_horizon = 10                                  # days ahead
t_ahead = np.arange(0, forecast_horizon + 1)
x_hat_mean = (rho**t_ahead) * x0
x_hat_var = sigma_w_hat**2 * (1 - rho**(2*t_ahead)) / (1 - rho**2)    # use THIS series' rho
x_hat_var[0] = 0
x_hat_std = np.sqrt(x_hat_var)

In [ ]:
#HW2 AR(1) forecast intervals
T0, k, sigma_w2, phi = 25, 0.001, 0.01, 0.75

def forecast(t):
    mean = T0 + k * t
    var  = sigma_w2 * (1 - phi**(2*t)) / (1 - phi**2)      # grows with t
    half = 1.96 * np.sqrt(var)
    return mean, (mean - half, mean + half)

var_infinity = sigma_w2 / (1 - phi**2)                      # long-run variance

# Climatology, anomalies and running means


In [ ]:
#Climatology and anomalies
monthly = df.groupby(["Year", "Month"])["TMAX_C"].mean().reset_index()   # one value per month
clim = monthly.groupby("Month")["TMAX_C"].agg(["mean", "std", "count"])   # average January, February, ...
monthly["anom"] = monthly["TMAX_C"] - monthly["Month"].map(clim["mean"])   # value minus its calendar month's mean

In [ ]:
#Plot anomalies
monthly["anom"].plot()
plt.ylabel('degree C')

In [ ]:
#One month's yearly series, with a completeness check
sub = df[df["Month"] == 7]                                     # keep July rows only
counts = sub.groupby("Year")["TMAX_C"].count()                  # number of July days per year
yearly = sub.groupby("Year")["TMAX_C"].mean().reset_index()     # July mean for each year
yearly = yearly[yearly["Year"].isin(counts[counts >= 20].index)].reset_index(drop=True)

In [ ]:
#12-month running mean
x = monthly["TMAX_C"].values
t_step = 12
n = len(x)
out = np.full(n, np.nan)             # start with all missing
h = t_step // 2                      # half-window = 6

for i in range(h, n - h):
    w = x[i-h : i+h+1].copy()        # the 13 values centered on month i
    if t_step % 2 == 0:              # even window: half weight on the two end values
        w[0] *= 0.5
        w[-1] *= 0.5
    out[i] = w.sum() / t_step

running_trend = pd.Series(out, index=monthly.index)
monthly['running_trend'] = running_trend

- A running mean replaces each value by the average of a window around it. It is a low-pass filter: it keeps slow changes and smooths away fast ones.
- A 12-month window averages over one whole seasonal cycle, so the seasonal cycle cancels out.
- With an even window the average needs a center month, so the notebook uses 13 points with half weight on the two ends (12 months total weight).
- The first and last 6 values are left as NaN because the window does not fit.
- Shortcut (added, not in your notebooks): monthly["TMAX_C"].rolling(12, center=True).mean(). It uses a plain 12-point window without the half weights, so results differ slightly from the loop.
- Use t_step = 31 for daily data. A longer window keeps only slower variability: to remove variability shorter than about 7 years, as in the Oregon exercise, a window of 84 months (7 years × 12) is the natural choice.

In [ ]:
#Seasonal cycle and anomaly from the running mean
# 1. detrended monthly value, averaged by calendar month = seasonal cycle
running_seasonal = (monthly["TMAX_C"] - monthly["running_trend"]).groupby(monthly["Month"]).mean()

# 2. add the mean level back so it is in degrees C rather than departures (HW2 does this)
running_seasonal_level = running_seasonal + monthly["running_trend"].mean()

# 3. residual = what is left after removing trend and seasonal cycle
seasonal_full = monthly["Month"].map(running_seasonal)
monthly['running_anomaly'] = monthly["TMAX_C"] - monthly["running_trend"] - seasonal_full

The running mean strips out the slow trend first, so the seasonal cycle left over is cleaner than the plain climatology (which still contains some warming). HW2 explains it this way: the climatology and running-mean seasonal cycles are very similar, with the running-mean version removing the 12-month trend before averaging by month. Subtracting both the running mean and the seasonal cycle leaves running_anomaly, the fast variability.

In [ ]:
#Oregon workflow (hands-on)
d = pd.read_csv("PRISM_ppt_tmean_45.0000_-123.0000_189501_202607.csv")
d["Date"] = pd.to_datetime(d["Date"], format="%Y-%m")
d["Year"], d["Month"] = d.Date.dt.year, d.Date.dt.month
d["T"] = (d["tmean (degrees F)"] - 32) * 5 / 9           # °F to °C

Then the order of steps in the lecture is: compute the climatology and anomalies as above, derive the long-term trend from the deseasonalized data (Section 4), then smooth with a running mean to remove interannual variability and fit the trend again. Your notebook runs the climatology, the anomaly, the 12-month running mean and the running anomaly. The trend fits on top use the Section 4 recipe, and a longer running-mean window (for example 84 months) is how you would filter out variability shorter than 7 years (not run in your notebook).

# Spectral analysis and band-pass filters

From Lecture 8. The power spectrum shows how much variance a series has at each timescale. You find the peaks, convert them to periods, then use a band-pass filter to pull out one of those timescales.

The one setting that controls the units: fs
fs is how many samples you have per unit of time. It sets the units of every frequency the functions return.
- Monthly data with fs = 12.0: frequencies come out in cycles per year.
- Daily data with fs = 365.0: frequencies also come out in cycles per year.
- The period is 1 divided by the frequency (in years). A peak at 0.25 cycles/year is a 4-year cycle.
- The highest frequency you can resolve (the Nyquist frequency) is fs / 2: 6 cycles/year for monthly data, which is a 2-month period.

In [ ]:
#Periodogram and Welch estimates
y = df["RONI"].values            # monthly, 1950 to 2026
fs = 12.0                        # samples per year

f_raw, P_raw = signal.periodogram(y, fs=fs)            # raw power at every resolvable frequency
f_w,  P_w  = signal.welch(y, fs=fs, nperseg=240)       # 20-year segments
f_w2, P_w2 = signal.welch(y, fs=fs, nperseg=120)       # 10-year segments

plt.semilogy(f_raw, P_raw, color="lightgray", label="raw periodogram")
plt.semilogy(f_w2, P_w2, "--", label="Welch, 10-yr segments")
plt.semilogy(f_w,  P_w,  lw=2.5, label="Welch, 20-yr segments")
plt.axvspan(1/7, 1/2, color="gold", alpha=0.15, label="ENSO band (2-7 yr)")
plt.xlabel("Frequency (cycles/year)"); plt.ylabel("PSD (degC^2 per cycle/yr)")

- periodogram returns the frequencies and the power at each one. It does the "compute every resolvable frequency at once" step from your notes, but the raw result is very noisy.
- welch cuts the series into segments of length nperseg, computes a periodogram for each and averages them. That smooths the noise but loses frequency resolution.
- Resolution is fs / nperseg: 12 / 240 = 0.05 cycles/year for 20-year segments, 0.1 for 10-year segments. Longer segments give finer resolution; shorter ones give a smoother curve.
- semilogy puts power on a log scale because it varies over orders of magnitude. axvspan shades the ENSO band: frequencies 1/7 to 1/2 cycles/year, which is periods of 2 to 7 years.

In [ ]:
#Hands-on: find the peaks, then filter
The "unknown X" series is daily (fs = 365.0). Your notes record three peaks at about 45 days, 4 years and 10 years.
from scipy.signal import find_peaks

x = df["x"].values
fs = 365.0
f, P = signal.periodogram(x, fs=fs)
pk, _ = find_peaks(P, height=P.max() * 0.05)     # indices of local maxima above a threshold (added)
print(f[pk], 1 / f[pk])                          # peak frequencies (cycles/yr) and periods (years)

In [ ]:
yr = df["day"].values / 365.0
band_1 = [365/60, 365/30]            # keep periods between 30 and 60 days (frequencies in cycles/year)

sos = signal.butter(4, band_1, btype="bandpass", fs=fs, output="sos")
x_1 = signal.sosfiltfilt(sos, x)
var_1 = 100 * x_1.var() / x.var()    # percent of the variance in this band

plt.plot(yr, x, label='raw')
plt.plot(yr, x_1, label=f"30-60 days {var_1:.2f}%")

In [ ]:
#The other two bands in  notebook, in the same pattern:
band_2 = [1/7, 1/3]     # periods 3 to 7 years (ENSO-like); notebook label says 3-7 yr
band_3 = [1/15, 1/3]    # periods 3 to 15 years as coded; the label says 8-15 yr

In [ ]:
#Harmonic fit by regression (added)
N = len(x)
t = np.arange(N) / 12                    # time in years for monthly data
k = 1                                    # wavenumber: cycles per year
H = np.column_stack([np.ones(N),
                     np.cos(2*np.pi*k*t),
                     np.sin(2*np.pi*k*t)])
fit = sm.OLS(x, H).fit()
print(fit.rsquared)

# Regression

From Lecture 9 and HW3. Your example predicts western US burned area (afab_km2) from vapor pressure deficit (vpd_hPa, how dry the air is), wind (wind_ms), temperature (temp_C) and relative humidity (rh_pct).

In [ ]:
#Fit a straight line with statsmodels
y = df['afab_km2']
X = sm.add_constant(df['vpd_hPa'])      # adds a column of ones so the model has an intercept
m1_lin = sm.OLS(y, X).fit()             # ordinary least squares
m1_lin.summary()                        # full table of results

In [ ]:
#Check the residuals for autocorrelation
stattools.acf(m1_lin.resid, nlags=1)[1]       # 0.07 in notebook

The regression assumes independent errors. A lag-1 autocorrelation of the residuals near 0 (0.07 here) means you do not need to worry; a large value means the standard errors are too small (use N_eff, as in Section 9, or sm.GLS).

In [ ]:
#Log-linear model (when the data curve)
m1_loglinear = sm.OLS(np.log(y), X).fit()     # predict log(burned area)
logy_fit = m1_loglinear.predict(xx_vpd)
y_fit = np.exp(logy_fit)                       # back to km^2

Burned area is skewed and grows faster than a straight line, so the notebook fits log(y) instead. The R² rises from 0.627 to 0.791. Taking np.exp of the prediction converts back to km² (it gives the median, not the mean).

In [ ]:
#To plot a smooth fitted line over a range of VPD values:
xx = np.linspace(x.min(), x.max(), 200)       # 200 evenly spaced VPD values
xx_vpd = sm.add_constant(xx)                  # add the intercept column to match the model
plt.plot(x, y, 'k.')
plt.plot(xx, m1_lin.predict(xx_vpd), 'b-', label=f"Linear: r2 = {m1_lin.rsquared:.2f}")
plt.plot(xx, np.exp(m1_loglinear.predict(xx_vpd)), 'r-', label=f"log-linear: r2 = {m1_loglinear.rsquared:.2f}")
plt.legend()

In [ ]:
#Adding predictors and comparing models
X = sm.add_constant(df[['vpd_hPa', 'wind_ms']])                      # two predictors
m2 = sm.OLS(np.log(y), X).fit()
print(f"R2 = {m2.rsquared:.3f}, adjusted R2 = {m2.rsquared_adj:.3f}")

 R² can never go down when you add a predictor, even a useless one. Adjusted R² can, so it is the fairer comparison.

In [ ]:
#Standardize predictors, using training statistics only (HW3)
train = np.arange(27)                   # 1984-2010
test  = np.arange(27, n)                # 2011-2020, n = len(df)

vpd_mean = df['vpd_hPa'].iloc[train].mean()
vpd_sd   = df['vpd_hPa'].iloc[train].std(ddof=1)

z_vpd_training = (df['vpd_hPa'].iloc[train] - vpd_mean) / vpd_sd
z_vpd_testing  = (df['vpd_hPa'].iloc[test]  - vpd_mean) / vpd_sd     # same training mean and sd

Repeat for wind and temperature. A z-score subtracts the mean and divides by the standard deviation, so every predictor is in units of "standard deviations" and the coefficients are directly comparable. The test years must use the training mean and SD. If you standardized them with their own statistics, information from the test period would leak in.

In [ ]:
#Fit, evaluate and interpret (HW3)
X = sm.add_constant(pd.DataFrame({'vpd_hPa': z_vpd_training,
                                  'wind_ms': z_wind_training,
                                  'temp_C':  z_temp_training}))
y_training = log_y[train]
m3 = sm.OLS(y_training, X).fit()

train_rmse = np.sqrt(np.mean((y_training - m3.predict(X))**2))

X_test = sm.add_constant(pd.DataFrame({'vpd_hPa': z_vpd_testing,
                                       'wind_ms': z_wind_testing,
                                       'temp_C':  z_temp_testing}))
y_test_predict = m3.predict(X_test)
y_testing = log_y[test]

test_rmse = np.sqrt(np.mean((y_testing - y_test_predict)**2))
test_r2 = np.corrcoef(y_testing, y_test_predict)[0, 1]**2

- RMSE (root mean squared error) is the typical size of a prediction error, in log units here: square each error, average, take the square root.
- Training versus testing: fit on 1984 to 2010 (27 years), then predict 2011 to 2020 without letting the model see them. Your results: training R² = 0.944 and RMSE = 0.213; testing RMSE = 0.247 and r² = 0.881. The testing error is a bit larger, which is normal because the model never saw those years.
- np.corrcoef(a, b)[0, 1]**2 squares the correlation between observed and predicted. That equals R² on the training data, but on new data it ignores any bias in the predictions.
- The fitted equation (standardized predictors): log(burned area) = 7.853 + 0.599 z_vpd + 0.408 z_wind + 0.120 z_temp.

In [ ]:
#Reading the coefficients. In a log model, a coefficient b means a one-unit (here one standard deviation) increase multiplies burned area by exp(b):
print(100 * (np.exp(0.5987) - 1))   # VPD: about +82% per 1 SD
print(100 * (np.exp(0.408) - 1))    # wind: about +50%
print(100 * (np.exp(0.120) - 1))    # temperature: about +13% (p = 0.065, not significant)

# Detection and attribution

From Lecture 10 and HW3. You regress the observed response on the model-simulated responses to each forcing. The coefficients are the scaling factors, and their confidence intervals tell you whether each forcing is detected and attributed.

In [ ]:
d = pd.read_csv("synthetic_DA_data.csv")
ctrl = pd.read_csv("synthetic_DA_control.csv")["control"]
d["fGHG"] = d["fALL"] - d["fAER"] - d["fNAT"]        # greenhouse-gas part = all minus the others

o = d["obs_proxy"]                                    # observed proxy (Y)
X = sm.add_constant(d[["fGHG", "fAER", "fNAT"]])      # the three forced responses (X) plus intercept

# 2. OLS fit
res = sm.OLS(o, X).fit()

# 3. effective sample size from the control run (noise only, no forcing)
n, p = len(o), len(res.params)
r1 = acf(ctrl, nlags=1)[1]                            # lag-1 autocorrelation of the noise
n_eff = n * (1 - r1) / (1 + r1)
t = stats.t.ppf(0.975, n_eff - p)                     # critical t with N_eff degrees of freedom

# 4. corrected confidence interval: beta +/- t * SE
beta = res.params.drop("const")                       # drop the intercept; keep only the scaling factors
se = res.bse.drop("const") * np.sqrt(n / n_eff)       # inflate the standard errors
CI_high = beta + t * se
CI_low  = beta - t * se

Reading the intervals
If the CI…
Then…
excludes 0
The forcing is detected
excludes 0 and includes 1
Detected and attributed (consistent): same pattern and amplitude as the model
includes 0
Not detected, so it cannot be attributed
excludes 1 and lies above it
Detected, but the model underestimates the response (β > 1)
excludes 1 and lies below it
Detected, but the model overestimates the response (β < 1)

In [ ]:
#HW3: potential intensity (PI)
#No natural-only run is provided, so the notebook builds it by subtraction:
d = pd.read_csv("NH_PI_DA_input.csv")
reg = d.dropna(subset=["year"]).copy()       # 1850-2005 (156 years)
control = d["NH_control"].dropna()           # the longer 500-year control run

reg["GHG"] = reg["NH_GHG"]
reg["AER"] = reg["NH_aerosols"]
reg["NAT"] = reg["NH_hist_aer"] - reg["NH_GHG"]    # hist_aer = GHG + NAT, so NAT = hist_aer - GHG
obs = reg["NH_obs_proxy"]

plt.plot(reg["year"], reg["NAT"]); plt.title('NAT forcing'); plt.ylabel('PI response')

In [ ]:
#Then the fit and correction are exactly the recipe above, with obs as Y and reg[["GHG", "AER", "NAT"]] as X:
X = sm.add_constant(reg[["GHG", "AER", "NAT"]])
m4 = sm.OLS(obs, X).fit()
n, p = len(obs), len(m4.params)
r1 = acf(control, nlags=1)[1]                 # 0.480
n_eff = n * (1 - r1) / (1 + r1)               # 54.8
t = stats.t.ppf(0.975, n_eff - p)
beta = m4.params.drop("const")
se = m4.bse.drop("const") * np.sqrt(n / n_eff)
CI_high, CI_low = beta + t * se, beta - t * se

In [ ]:
#Attributable change
period = reg[(reg["year"] >= 1850) & (reg["year"] <= 2005)]
for forcing in ['GHG', 'AER', 'NAT']:
    slope, intercept = np.polyfit(period["year"], period[forcing], 1)   # linear trend of the modeled response
    change = beta[forcing] * slope * (2005 - 1850)                       # scaled trend x number of years
    print(forcing, "trend =", slope, "attributable change =", change)

slope_obs, _ = np.polyfit(period["year"], period["NH_obs_proxy"], 1)
print("Observed change =", slope_obs * 155)